# 04 · LangGraph 入门

> 学习目标：理解为什么需要 LangGraph，掌握 State / Node / Edge 三个核心概念。

## 知识点
- 为什么需要 LangGraph：Agent 逻辑有循环和分支，普通 Chain（单向流水线）表达不了
- 三大核心概念：
  - `State`（状态）— 整张图共享的"记事本"
  - `Node`（节点）— 一个个步骤（函数），读状态、干活、更新状态
  - `Edge`（边）— 决定下一步走到哪个节点
- `create_react_agent`：官方预构建的 ReAct 智能体，一键创建

## 一、先用预构建的：create_react_agent（10 行代码的 Agent）

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 自定义一个工具
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气。"""
    return f'{city}今天晴，25°C'

model = init_chat_model('deepseek-chat', model_provider='deepseek')

# ⭐ 一行创建带工具的 Agent（本质是一个 LangGraph 图）
agent = create_react_agent(model, tools=[get_weather])

result = agent.invoke({'messages': [('user', '北京今天天气怎么样？')]})
for msg in result['messages']:
    print(type(msg).__name__, ':', msg.content)

## 二、手写一个最小的图：理解 State / Node / Edge

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ⭐ State：定义整张图共享的状态结构
class State(TypedDict):
    text: str

# ⭐ Node：每个节点是一个函数，接收状态、返回更新
def upper_node(state: State):
    print('节点1：把文本变大写')
    return {'text': state['text'].upper()}

def add_suffix(state: State):
    print('节点2：加后缀')
    return {'text': state['text'] + ' —— 已处理'}

# ⭐ Edge：把节点连成图
graph = StateGraph(State)
graph.add_node('upper', upper_node)
graph.add_node('suffix', add_suffix)
graph.add_edge(START, 'upper')       # 入口 -> 节点1
graph.add_edge('upper', 'suffix')    # 节点1 -> 节点2
graph.add_edge('suffix', END)        # 节点2 -> 出口

app = graph.compile()
print(app.invoke({'text': 'hello langgraph'}))

## 三、条件边：让图会"分岔"（Agent 决策的本质）

In [ ]:
# ⭐ 条件边 = 根据状态决定走哪条路，Agent 的"思考-行动-观察"循环就是靠它实现的
# 预告：ReAct Agent 内部就是一个这样的循环图：
#   agent(思考) --要不要用工具?--> tools(行动) --> agent(观察结果再思考) --> ... --> END

# TODO: 尝试给上面的图加一个条件边：
#   text 长度超过 10 走节点A，否则走节点B（用 graph.add_conditional_edges）

## 常见坑
- Node 返回的是**增量更新**字典，不是整个 State
- 图必须 `compile()` 之后才能 `invoke`
- `create_react_agent` 适合快速起步；自定义业务流程再手写 StateGraph
- 中文文档：https://langchain-ai.github.io/langgraph/